# M3 B — Eclat: kiểm chứng và bàn giao

Notebook Python gọi Julia 1.9+ của nhóm. **Runtime → Run all** tải nguồn có kiểm hash,
chạy full Julia tests, hai ứng dụng, bốn benchmark đối chiếu list/BitVector/Java SPMF,
và kiểm PDF/số liệu bàn giao. Kết quả ứng dụng được mine mới trong từng phiên.
RUN_BENCHMARKS mặc định tắt: đây là tùy chọn đo lại runtime/RSS trên Linux/macOS;
kiểm chứng số liệu đo đã lưu và bốn full maps luôn bật.

Colab dùng Java 21 và Julia có sẵn hoặc bản chính thức kiểm checksum. Local chọn kernel Python 3.11+.
M3 chưa push: upload notebook này và Git bundle eclat-m3-source.bundle vào /content trước Run all.
Sau khi M3 được publish, có thể chạy trực tiếp từ public main. Phiên dùng bundle được ghi rõ là bản local,
không được dùng làm bằng chứng public main đã chứa M3.


In [1]:
from pathlib import Path
import csv
import json
import tempfile
import time
import os
import sys
import hashlib
import re
import shutil
import subprocess
import urllib.request

def run_command(command, **kwargs):
    # Forward captured process output into the notebook before raising on errors.
    # Colab/Jupyter may otherwise show only CalledProcessError, hiding Julia's log.
    check = kwargs.pop("check", True)
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding="utf-8", errors="replace", **kwargs)
    if result.stdout:
        print(result.stdout, end="", flush=True)
    if check:
        result.check_returncode()
    return result

REPO_URL = "https://github.com/phuctran33/-DM-Eclat.git"
REPO_REF = os.environ.get("ECLAT_REPO_REF", "main")
RUN_PREPARATION = True
JULIA_VERSION = "1.11.4"
RUN_MINER = True
RUN_SPMF = True
RUN_OPTIMIZED = True
RUN_APPLICATIONS = True
RUN_BENCHMARKS = False
RUN_BENCHMARK_VERIFICATION = True
FINAL_DELIVERY_VALIDATION = True
SEED = 42

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content") / ("eclat-m3-" + str(time.time_ns()))
    if not (ROOT / "Project.toml").exists():
        if not REPO_URL.startswith("https://github.com/"):
            raise ValueError("Điền URL GitHub public thật của nhóm trước khi chạy Colab.")
        # Optional draft source bundle: lets a reviewer cold-run local M3 before publication.
        bundle = Path(os.environ.get("ECLAT_SOURCE_BUNDLE", "/content/eclat-m3-source.bundle"))
        if bundle.is_file():
            run_command(["git", "clone", "--branch", "feat/member-b-m3-delivery", str(bundle), str(ROOT)], check=True)
            print("SOURCE: uploaded local Git bundle; this is not a public GitHub M3 run.")
        else:
            run_command(["git", "clone", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
    if not shutil.which("julia"):
        archive = Path("/content/julia-runtime.tar.gz")
        runtime = Path("/content/julia-runtime")
        runtime.mkdir(exist_ok=True)
        urllib.request.urlretrieve(
            f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz",
            archive,
        )
        checksum_url = f"https://julialang-s3.julialang.org/bin/checksums/julia-{JULIA_VERSION}.sha256"
        checksum_text = urllib.request.urlopen(checksum_url, timeout=30).read().decode("utf-8")
        filename = f"julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
        expected = next(line.split()[0] for line in checksum_text.splitlines() if line.split()[-1].lstrip("*") == filename)
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Julia archive checksum mismatch"
        run_command(["tar", "-xzf", str(archive), "-C", str(runtime), "--strip-components=1"], check=True)
        os.environ["PATH"] = str(runtime / "bin") + os.pathsep + os.environ["PATH"]
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "Project.toml").exists()), None)
    if ROOT is None:
        raise RuntimeError("Mở notebook từ repo root hoặc notebooks/.")

os.chdir(ROOT)
os.environ.setdefault("JULIA_DEPOT_PATH", str(ROOT / ".tools/julia-depot"))
JULIA = os.environ.get("ECLAT_JULIA") or shutil.which("julia")
if not JULIA:
    candidates = sorted((ROOT / ".tools").glob("julia-*/bin/julia.exe"))
    JULIA = str(candidates[-1]) if candidates else None
if not JULIA:
    raise RuntimeError("Cài Julia 1.9+ và thêm vào PATH khi chạy local.")

def run_julia(*arguments, env=None):
    return run_command(
        [JULIA, "--startup-file=no", f"--project={ROOT}", *map(str, arguments)],
        cwd=ROOT, env={**os.environ, **(env or {})}, check=True,
    )

version = subprocess.check_output([JULIA, "--version"], text=True)
match = re.search(r"(\d+)\.(\d+)", version)
assert match and tuple(map(int, match.groups())) >= (1, 9), "Julia 1.9+ required"
assert (ROOT / "scripts/prepare_applications.jl").is_file(), "Checkout phải có code B; chọn đúng nhánh đã publish hoặc mở source B local."
run_julia("--version")
run_julia("-e", "using Pkg; Pkg.instantiate()")
# Respect an explicit executable; otherwise find an existing Java >=21 locally.
def java_major(executable):
    try:
        result = subprocess.run([str(executable), "-version"], capture_output=True,
                                text=True, timeout=15)
    except (OSError, subprocess.TimeoutExpired):
        return 0
    match = re.search(r'(?:openjdk|java)(?:\s+version)?\s+"?(\d+)(?:\.(\d+))?',
                      result.stdout + result.stderr)
    if result.returncode or not match:
        return 0
    return int(match[2]) if match[1] == "1" and match[2] else int(match[1])

JAVA = os.environ.get("ECLAT_JAVA") or shutil.which("java")
if RUN_SPMF and not IN_COLAB and not os.environ.get("ECLAT_JAVA"):
    java_candidates = [JAVA] if JAVA else []
    for pattern in ["jdk*/bin/java.exe", "java*/bin/java.exe", "jdk*/bin/java", "java*/bin/java"]:
        java_candidates.extend((ROOT / ".tools").glob(pattern))
    if os.name == "nt":
        for directory in [os.environ.get("ProgramFiles"), os.environ.get("ProgramFiles(x86)")]:
            if directory:
                for pattern in ["Java/*/bin/java.exe", "Eclipse Adoptium/*/bin/java.exe", "JetBrains/*/jbr/bin/java.exe"]:
                    java_candidates.extend(Path(directory).glob(pattern))
        java_candidates.extend(Path(ROOT.anchor).glob("CLion*/jbr/bin/java.exe"))
    JAVA = next((str(path) for path in java_candidates if java_major(path) >= 21), None)
if RUN_SPMF:
    if IN_COLAB:
        run_command(["apt-get", "-qq", "update"], check=True)
        run_command(["apt-get", "-qq", "install", "-y", "openjdk-21-jre-headless"], check=True)
        JAVA = "/usr/lib/jvm/java-21-openjdk-amd64/bin/java"
    if not JAVA or java_major(JAVA) < 21:
        raise RuntimeError("Java >=21 required for live SPMF. Set ECLAT_JAVA to its executable.")
    java_version = subprocess.run([JAVA, "-version"], capture_output=True, text=True, check=True)
    print("JAVA:", JAVA)
    print(java_version.stderr.strip())
    run_command([sys.executable, "scripts/prepare_spmf.py", "--java", JAVA], check=True)
print("ROOT:", ROOT, "SEED:", SEED)
import tempfile
work_parent = ROOT / ".tools/notebook-runs"
work_parent.mkdir(parents=True, exist_ok=True)
NOTEBOOK_OUTPUT = Path(tempfile.mkdtemp(prefix="integrated-", dir=work_parent))


Cloning into '/content/eclat-m3-1794915521600565231'...
SOURCE: uploaded local Git bundle; this is not a public GitHub M3 run.
julia version 1.12.6
Precompiling packages...
 248059.2 ms  ✓ Pkg (serial)
  1 dependency successfully precompiled in 248 seconds
  Installing known registries into `/content/eclat-m3-1794915521600565231/.tools/julia-depot`
       Added `General` registry to /content/eclat-m3-1794915521600565231/.tools/julia-depot/registries
┌ Warning: The active manifest file has dependencies that were resolved with a different julia version (1.10.10). Unexpected behavior may occur.
└ @ /content/eclat-m3-1794915521600565231/Manifest.toml:0
┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.
└ @ Pkg.API /usr/local/share/julia/stdlib/v1.12/Pkg/src/API.jl:1322
Precompiling packages...
    585.4 ms  ✓ Unicode
    753.8 ms  ✓ EclatProject
  2 dep

## B: dữ liệu và audit

Nguồn/hash và toàn bộ preprocessing đã được kiểm chéo độc lập. Cell tiếp theo
chuẩn bị dữ liệu thật; dữ liệu này chưa phải kết quả ứng dụng mined cuối cùng.


In [2]:
if RUN_PREPARATION:
    import tempfile
    run_command([sys.executable, "scripts/prepare_application_data.py", "--dataset", "all"], check=True)
    work_parent = ROOT / ".tools/notebook-runs"
    work_parent.mkdir(parents=True, exist_ok=True)
    B_OUTPUT = Path(tempfile.mkdtemp(prefix="member-b-", dir=work_parent))
    run_julia("scripts/prepare_applications.jl", "groceries", "data/applications/groceries.csv", B_OUTPUT / "groceries")
    run_julia("scripts/prepare_applications.jl", "splice", "data/applications/splice.data", B_OUTPUT / "splice")
    run_command([sys.executable, "-B", "tests/verify_prepared_data.py", "--julia", JULIA,
                    "--report", str(B_OUTPUT / "preparation-report.json")], check=True)
    print("B preprocessing artifacts:", B_OUTPUT)
else:
    print("B preprocessing không chạy trong phiên này.")


{"dataset": "groceries", "source_url": "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/d20658ec6d336af2d4ddb5fd72b6f677dd46136e/groceries.csv", "source_revision": "d20658ec6d336af2d4ddb5fd72b6f677dd46136e", "source_sha256": "ff1be892fd6b9b57d1a7bc50de067798963dda607619645988b21789bf23ae3b", "adapter": "one source CSV row = one basket; long CSV, preserve every item", "n_source_baskets": 9835, "output_sha256": "2c4beb856d64e04ecd57844e662889e6a60b7a9519b3ee0e5d98078c694f41d7"}
{"dataset": "splice", "source_url": "https://www.inf.u-szeged.hu/~tothl/gepitan/uci%20repository/molecular-biology/splice-junction-gene-sequences/splice.data", "source_revision": "sha256:ebbae10c85d3c285e2a2489a37e42bc2ee8d91d15005e391a9962fa3389a114f", "original_source": "https://doi.org/10.24432/C5M888", "original_download_url": "https://archive.ics.uci.edu/static/public/69/molecular+biology+splice+junction+gene+sequences.zip", "source_kind": "University of Szeged academic UCI mirror; or

## Kiểm thử chung sau tích hợp

Full suite bật miner list/bitset, rules, CLI, SPMF và toàn bộ checks B.


In [3]:
run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1", "ECLAT_TEST_BACKENDS": "list,bitset"})
run_command([sys.executable, "-B", "tests/test_application_sources.py"], check=True)
run_command([sys.executable, "-B", "tests/test_delivery.py", "-v"], check=True)
run_julia("scripts/mine.jl", "--help")


Precompiling packages...
  14839.0 ms  ✓ Test (serial)
  1 dependency successfully precompiled in 15 seconds
Test Summary:                     | Pass  Total  Time
SPMF I/O and canonical comparison |   14     14  3.6s
Test Summary:                           | Pass  Total  Time
Threshold boundaries and API validation |   19     19  1.4s
Test Summary:    | Pass  Total  Time
Shared schema v1 |   33     33  3.4s
Test Summary:                      | Pass  Total  Time
B: CSV and Groceries normalization |   18     18  2.0s
Test Summary:                                            | Pass  Total  Time
B: Splice normalization and independent class comparison |   24     24  1.9s
Test Summary:                                          | Pass  Total  Time
B: RFC4180 writer, record positions and rule artifacts |   12     12  1.9s
Test Summary:                                          | Pass  Total  Time
B: reject inconsistent DNA patterns and export ranking |    7      7  0.7s
Test Summary:            

CompletedProcess(args=['/usr/local/bin/julia', '--startup-file=no', '--project=/content/eclat-m3-1794915521600565231', 'scripts/mine.jl', '--help'], returncode=0, stdout='Eclat frequent itemset miner\nUsage:\n  julia --project=. scripts/mine.jl --input FILE --output FILE --minsup 0.4 [--backend list|bitset]\n  julia --project=. scripts/mine.jl --input FILE --output FILE --min-count 2 [--backend list|bitset]\nAccepts relative decimals or percentages, e.g. 0.4 or 40%.\nExactly one of --minsup / --min-count is required.\n\n')

## Miner Basic và references SPMF

CLI chạy code A đã merge. References A/B chạy độc lập và có cùng hashes.
Bật RUN_SPMF nếu cần tái tạo bằng Java >=21; ECLAT_JAVA chọn executable.


In [4]:
if RUN_MINER:
    run_julia("scripts/mine.jl", "--input", "data/toy/basic.txt", "--output", NOTEBOOK_OUTPUT / "basic.txt",
              "--minsup", "40%", "--backend", "list")
    run_julia("scripts/compare_spmf.jl", NOTEBOOK_OUTPUT / "basic.txt", "tests/reference/spmf/basic.txt")
if RUN_SPMF:
    print("JAR SPMF đã kiểm SHA-256; live verification hai ứng dụng thực hiện ở cell tiếp theo.")


transactions=5, min_count=2, patterns=7
reference=7, own=7, exact_matches=7
missing=0, extra=0, wrong_support=0
exact_match_rate=1.0; identical=true
JAR SPMF đã kiểm SHA-256; live verification hai ứng dụng thực hiện ở cell tiếp theo.


## M2: hai ứng dụng từ dữ liệu thật

Groceries: 9.835 giỏ, 169 items; minsup=0.005 (count=50), minconf=0.60.
Hai backend nhóm phải có cùng toàn bộ map, rồi sinh mọi luật đủ ngưỡng và lấy top-10 theo lift.
Splice: window 25:36, 48 IDs; giữ 3.185/3.190 mẫu. Nhãn EI/IE/N không đưa vào miner.
Minsup=0.20 riêng từng lớp; mine đầy đủ rồi hợp candidates 2–4 item và đếm lại mọi lớp.
SPMF kiểm cả Groceries và ba lớp bằng cùng absolute count, không giới hạn output.


In [5]:
if RUN_OPTIMIZED:
    run_julia("scripts/mine.jl", "--input", "data/toy/basic.txt", "--output", NOTEBOOK_OUTPUT / "bitset.txt",
              "--minsup", "40%", "--backend", "bitset")
    run_julia("scripts/compare_spmf.jl", NOTEBOOK_OUTPUT / "bitset.txt", "tests/reference/spmf/basic.txt")

if RUN_APPLICATIONS:
    APPLICATION_OUTPUT = NOTEBOOK_OUTPUT / "applications"
    run_julia("scripts/run_applications.jl", "--output", APPLICATION_OUTPUT)
    command = [sys.executable, "-B", "scripts/verify_application_results.py", "--output", str(APPLICATION_OUTPUT)]
    if RUN_SPMF:
        command += ["--java", JAVA]
    run_command(command, check=True)
    validation = json.loads((APPLICATION_OUTPUT / "validation.json").read_text(encoding="utf-8"))
    print("Ứng dụng đã kiểm:", validation["n_rules"], "luật;", validation["n_dna_candidates"], "DNA patterns")
    def show_csv(path, limit=10):
        with path.open(encoding="utf-8", newline="") as stream:
            rows = list(csv.DictReader(stream))[:limit]
        try:
            import pandas as pd
            from IPython.display import display
            display(pd.DataFrame(rows))
        except ImportError:
            for row in rows:
                print(row)
    show_csv(APPLICATION_OUTPUT / "groceries_top10.csv")
    show_csv(APPLICATION_OUTPUT / "dna_patterns.csv")
    print("Artifacts:", APPLICATION_OUTPUT)
else:
    print("RUN_APPLICATIONS đang tắt trong phiên này.")


transactions=5, min_count=2, patterns=7
reference=7, own=7, exact_matches=7
missing=0, extra=0, wrong_support=0
exact_match_rate=1.0; identical=true
Groceries: 1001 frequent patterns, 22 rules, 10 top rules; min_count=50
Splice EI: 381 patterns; min_count=153
Splice IE: 209 patterns; min_count=154
Splice N: 48 patterns; min_count=331
DNA comparisons: 463; output=/content/eclat-m3-1794915521600565231/.tools/notebook-runs/integrated-han60929/applications
SPMF PASS groceries 1001
SPMF PASS splice_EI 381
SPMF PASS splice_IE 209
SPMF PASS splice_N 48
PASS independent complete rule/DNA count audit: 22 rules; 463 DNA candidates; 864 nonzero infrequent class counts
Ứng dụng đã kiểm: 22 luật; 463 DNA patterns


,rank,antecedent_ids,consequent_ids,antecedent_labels,consequent_labels,length,support_count,antecedent_count,consequent_count,n_transactions,support,confidence,lift
0,1,30 124 167,103,citrus fruit | root vegetables | whole milk,other vegetables,4,57,90,1903,9835,0.005795627859684799,0.6333333333333333,3.2731651777894553
1,2,110 124 167,103,pip fruit | root vegetables | whole milk,other vegetables,4,54,88,1903,9835,0.005490594814438231,0.6136363636363636,3.1713681746524625
2,3,110 163,103,pip fruit | whipped/sour cream,other vegetables,3,55,91,1903,9835,0.005592272496187087,0.6043956043956044,3.12361049355269
3,4,100 124,103,onions | root vegetables,other vegetables,3,56,93,1903,9835,0.0056939501779359435,0.6021505376344086,3.11200763932444
4,5,124 158 168,167,root vegetables | tropical fruit | yogurt,whole milk,4,56,80,2513,9835,0.0056939501779359435,0.7,2.739554317548746
5,6,103 110 124,167,other vegetables | pip fruit | root vegetables,whole milk,4,54,80,2513,9835,0.005490594814438231,0.675,2.6417130919220053
6,7,15 163,167,butter | whipped/sour cream,whole milk,3,66,100,2513,9835,0.006710726995424504,0.66,2.5830083565459607
7,8,110 163,167,pip fruit | whipped/sour cream,whole milk,3,59,91,2513,9835,0.005998983223182512,0.6483516483516484,2.537420796473721
8,9,15 168,167,butter | yogurt,whole milk,3,92,144,2513,9835,0.009354346720894764,0.6388888888888888,2.500386877127824
9,10,15 124,167,butter | root vegetables,whole milk,3,81,127,2513,9835,0.008235892221657347,0.6377952755905512,2.4961068585089814


,rank,pattern_ids,pattern_labels,length,count_EI,count_IE,count_N,n_EI,n_IE,n_N,support_EI,support_IE,support_N,support_span
0,1,27 32,pos=+1;base=G | pos=+2;base=T,2,758,108,98,765,766,1654,0.9908496732026144,0.1409921671018277,0.0592503022974607,0.9315993709051538
1,2,17 23,pos=-2;base=A | pos=-1;base=G,2,388,762,127,765,766,1654,0.5071895424836601,0.9947780678851175,0.07678355501813784,0.9179945128669796
2,3,27 32 43,pos=+1;base=G | pos=+2;base=T | pos=+5;base=G,3,651,22,15,765,766,1654,0.8509803921568627,0.028720626631853787,0.009068923821039904,0.8419114683358229
3,4,27 43,pos=+1;base=G | pos=+5;base=G,2,655,71,102,765,766,1654,0.8562091503267973,0.0926892950391645,0.06166868198307134,0.794540468343726
4,5,32 43,pos=+2;base=T | pos=+5;base=G,2,651,46,94,765,766,1654,0.8509803921568627,0.06005221932114883,0.056831922611850064,0.7941484695450126
5,6,23 27 32,pos=-1;base=G | pos=+1;base=G | pos=+2;base=T,3,618,108,25,765,766,1654,0.807843137254902,0.1409921671018277,0.015114873035066506,0.7927282642198354
6,7,14 17 23,pos=-3;base=C | pos=-2;base=A | pos=-1;base=G,3,174,604,38,765,766,1654,0.22745098039215686,0.7885117493472585,0.022974607013301087,0.7655371423339574
7,8,23 32,pos=-1;base=G | pos=+2;base=T,2,618,249,86,765,766,1654,0.807843137254902,0.32506527415143605,0.05199516324062878,0.7558479740142732
8,9,14 17,pos=-3;base=C | pos=-2;base=A,2,195,604,101,765,766,1654,0.2549019607843137,0.7885117493472585,0.06106408706166868,0.7274476622855899
9,10,23 27,pos=-1;base=G | pos=+1;base=G,2,624,363,146,765,766,1654,0.8156862745098039,0.47389033942558745,0.08827085852478839,0.7274154159850155


Artifacts: /content/eclat-m3-1794915521600565231/.tools/notebook-runs/integrated-han60929/applications


## Diễn giải kết quả ở cấu hình đã chốt

- Groceries có 1.001 tập phổ biến và 22 luật. Luật citrus fruit + root vegetables + whole milk → other vegetables có 57/90 giỏ thỏa hậu kiện, confidence 63,33%, lift 3,273; có thể thử bố trí/gợi ý mua kèm nhóm rau.
- Root vegetables + tropical fruit + yogurt → whole milk: 56/80 = 70%, lift 2,740. Butter + yogurt → whole milk: 92/144 = 63,89%, lift 2,500. Đây là gợi ý thử nghiệm bán chéo; không suy luận khuyến mãi gây tăng mua hàng từ dữ liệu quan sát.
- DNA có 463 candidates. (+1,G),(+2,T): EI 758/765=99,08%, IE 108/766=14,10%, N 98/1654=5,93%. (-2,A),(-1,G): EI 388/765=50,72%, IE 762/766=99,48%, N 127/1654=7,68%.
- Đây là co-occurrence mô tả trong mẫu, không phải accuracy classifier, p-value hay kết luận nhân quả sinh học. Lớp N đông hơn; dùng support riêng mỗi lớp. Cửa sổ/ngưỡng/ambiguity và việc chọn pattern trên cùng dữ liệu giới hạn khả năng khái quát.

CSV/config/run metadata là nguồn số liệu; xem docs/M2_MEMBER_B.md cho đầy đủ bảng, counts và lệnh tái lập.


## Kiểm chứng benchmark và tùy chọn đo lại

RUN_BENCHMARK_VERIFICATION mặc định bật: audit 35 cases/875 runs/70 comparisons của A,
kiểm 46 artifact hashes rồi chạy list/BitVector/SPMF trên bốn benchmark tại memory_minsup.
Đối chiếu toàn map; đây là correctness review, không phải đo lại runtime/RSS.
RUN_BENCHMARKS mặc định tắt, chỉ dùng nếu muốn đo lại trên Linux/macOS với output mới.
Xem docs/experiments.md về timing scopes và peak RSS.


In [6]:
if RUN_BENCHMARK_VERIFICATION:
    assert RUN_SPMF, "Benchmark full-map verification needs Java SPMF."
    run_command([sys.executable, "-B", "scripts/acquire_benchmarks.py"], check=True)
    BENCHMARK_REVIEW_OUTPUT = NOTEBOOK_OUTPUT / "benchmark-review"
    run_command([sys.executable, "-B", "scripts/review_benchmarks.py",
                 "--julia", JULIA, "--java", JAVA, "--output", str(BENCHMARK_REVIEW_OUTPUT)], check=True)

if RUN_BENCHMARKS:
    # Real A pipeline; Java >=21, Python >=3.11. Full grid can take several minutes.
    run_command([sys.executable, "scripts/prepare_spmf.py", "--java", JAVA], check=True)
    run_command([sys.executable, "scripts/acquire_benchmarks.py"], check=True)
    run_julia("scripts/benchmark_runner.jl", "--java", JAVA, "--output", str(NOTEBOOK_OUTPUT / "benchmark"))
    run_command([sys.executable, "-m", "pip", "install", "-r", "requirements-plot.txt"], check=True)
    run_command([sys.executable, "scripts/plot_results.py", "--results", str(NOTEBOOK_OUTPUT / "benchmark")], check=True)
else:
    print("Không đo lại runtime/RSS trong phiên này; bằng chứng đo của A được audit ở trên.")


chess {'dataset': 'chess', 'n_transactions': 3196, 'n_items': 75, 'average_width': 37.0, 'max_width': 37, 'density': 0.49333333333333335, 'input_sha256': '341d0ff8676e053dd3b417d61f84c732736154f987ce56975a099b07460957f5'}
mushroom {'dataset': 'mushroom', 'n_transactions': 8124, 'n_items': 119, 'average_width': 23.0, 'max_width': 23, 'density': 0.19327731092436976, 'input_sha256': '06206c73983251a9488414c4379a030e836bbeb6fe9eff1b5624a0193a811640'}
retail {'dataset': 'retail', 'n_transactions': 88162, 'n_items': 16470, 'average_width': 10.305755314080896, 'max_width': 76, 'density': 0.000625728920102058, 'input_sha256': '417563fb5feb3711d4f761230ca78b76d100fe2ee0d3178fcc4fbb000d8d1c36'}
T10I4D100K {'dataset': 'T10I4D100K', 'n_transactions': 100000, 'n_items': 870, 'average_width': 10.10228, 'max_width': 29, 'density': 0.011611816091954023, 'input_sha256': 'fb2dd118ff822ff88d724e47d5a4cea6531c0bd6294397db4d36890c1450f1a7'}
PASS frozen measurement source: 49dafc999b6c00f8c4275ee68f99d4a6e3

## Final cold-run — B chủ trì, A review

FINAL_DELIVERY_VALIDATION mặc định bật: kiểm PDF hiện hành/nguồn/số trang, bảng CSV,
benchmark evidence và các output ứng dụng mine mới trong phiên này cùng live SPMF.
Không cần bật RUN_BENCHMARKS để nghiệm thu correctness.
Reviewer rà nội dung PDF, biểu đồ/metrics và public links. Sau publish cần chạy lại
public main; phiên upload bundle chỉ chứng minh bản local hoạt động trên Colab.


In [7]:
if FINAL_DELIVERY_VALIDATION:
    assert all([RUN_PREPARATION, RUN_MINER, RUN_SPMF, RUN_OPTIMIZED,
                RUN_APPLICATIONS, RUN_BENCHMARK_VERIFICATION]), "Bật đủ các bước kiểm chứng bàn giao."
    run_command([sys.executable, "-B", "scripts/verify_delivery.py",
                 "--applications", str(APPLICATION_OUTPUT),
                 "--benchmark-review", str(BENCHMARK_REVIEW_OUTPUT),
                 "--output", str(NOTEBOOK_OUTPUT / "delivery-validation.json")], check=True)
    print("PASS M3: PDF hiện hành, số liệu A nguyên vẹn, bốn benchmark và bốn ứng dụng khớp SPMF.")
    print("Bằng chứng phiên chạy:", NOTEBOOK_OUTPUT / "delivery-validation.json")
else:
    print("FINAL_DELIVERY_VALIDATION đang tắt; phiên này chưa nghiệm thu đầy đủ M3.")


PASS independent complete rule/DNA count audit: 22 rules; 463 DNA candidates; 864 nonzero infrequent class counts
PASS final delivery: {"status": "pass", "runtime": "google-colab", "report_pdf_sha256": "99c89ffa5da9d2feb49da347f83f454f75e661ebe3b6eae32074cefe42e214c1", "report_body_pages": 10, "n_rules": 22, "n_dna_candidates": 463, "archived_benchmark_comparisons": 70, "live_benchmark_datasets": 4, "live_application_datasets": 4, "benchmark_review_sha256": "e68a10bb7e36ab0d5e4c96a03168431c15ec270ba15f6bd5d3b1bcf6c0424032", "application_validation_sha256": "6bbeaa7244784082c4d758c8806b8544d5a86e2891e61b0ab6c0541307151d10"}
PASS M3: PDF hiện hành, số liệu A nguyên vẹn, bốn benchmark và bốn ứng dụng khớp SPMF.
Bằng chứng phiên chạy: /content/eclat-m3-1794915521600565231/.tools/notebook-runs/integrated-han60929/delivery-validation.json
